# Người 3 — LightGBM, E2, E7 và SHAP

**Dự đoán tại thời điểm đặt phòng.** Chỉ dùng `train` để fit; dùng `val` để đánh giá; không mở `test`. Chạy `Restart & Run All` từ thư mục `notebooks/`.

- E1: LightGBM trên toàn bộ train; ghi xác suất theo đúng thứ tự val.
- E2: so sánh bản gốc, `class_weight=balanced`, và SMOTENC chỉ trên train. Với dữ liệu mã hóa thứ tự, dùng **SMOTENC** để các cột phân loại không bị nội suy thành mã không hợp lệ.
- E7: mô hình mới fit trên 80% đầu của train theo ngày đến, 20% cuối của train dùng riêng để hiệu chỉnh sigmoid. Cả hai dự đoán trên **cùng tập val** để so sánh công bằng. Đây là mô hình riêng, khác mô hình E1 được fit trên toàn train.
- SHAP: giải thích LightGBM E1 trên mẫu cố định của val. SHAP mô tả mối liên hệ mà mô hình đã học, không chứng minh nguyên nhân.

Cần `pip install -r requirements.txt`. Kết quả nằm trong `reports/results/nguoi3.csv`, `reports/figures/`, `models/`.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.pipeline import Pipeline

from src.calibration import fit_calibrator
from src.data import REPO_ROOT, SEED, load_xy
from src.evaluate import evaluate, evaluate_by_group, log_result, save_proba
from src.features import CATEGORICAL, NUMERIC, build_preprocessor, check_columns

AUTHOR = "nguoi3"  # đổi thành tên không dấu nếu nhóm muốn
FIGURES = REPO_ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
(REPO_ROOT / "models").mkdir(exist_ok=True)

def classifier(**kwargs):
    return LGBMClassifier(n_estimators=250, learning_rate=0.05, num_leaves=31,
                          max_depth=-1, random_state=SEED, n_jobs=2,
                          verbosity=-1, **kwargs)

def metrics_with_brier(y, p):
    return {**evaluate(y, p), "brier": brier_score_loss(y, p),
            "log_loss": log_loss(y, p)}


## E1 — Mô hình LightGBM trên toàn bộ train

In [ ]:
X_train, y_train = load_xy("train")
X_val, y_val = load_xy("val")
check_columns(X_train)
check_columns(X_val)
assert X_train.columns.tolist() == X_val.columns.tolist()
print(f"train: {len(X_train):,}, tỷ lệ hủy {y_train.mean():.2%}")
print(f"val: {len(X_val):,}, tỷ lệ hủy {y_val.mean():.2%}")

model = Pipeline([("prep", build_preprocessor("tree")), ("clf", classifier())])
model.fit(X_train, y_train)
p_e1 = model.predict_proba(X_val)[:, 1]
m_e1 = metrics_with_brier(y_val, p_e1)
log_result(AUTHOR, "E1", "lightgbm", "val", m_e1, "default")
save_proba("lightgbm", p_e1)
joblib.dump(model, REPO_ROOT / "models" / "lightgbm.joblib")
pd.Series(m_e1).round(4)

## E2 — Có cần xử lý mất cân bằng không?

Đánh giá trên cùng val và cùng ngưỡng 0,5. `class_weight` và SMOTENC chỉ dùng trong quá trình fit train; **không SMOTE val**. Vì đổi tỷ lệ lớp khi học có thể làm sai xác suất, so sánh thêm Brier và log loss.

In [ ]:
weighted = Pipeline([("prep", build_preprocessor("tree")),
                     ("clf", classifier(class_weight="balanced"))])
weighted.fit(X_train, y_train)
p_weighted = weighted.predict_proba(X_val)[:, 1]
m_weighted = metrics_with_brier(y_val, p_weighted)
log_result(AUTHOR, "E2", "lightgbm", "val", m_weighted, "class_weight=balanced")
save_proba("lightgbm_balanced", p_weighted)

# ColumnTransformer xuất số trước, chữ sau; SMOTENC chỉ tổng hợp các mã chữ theo nhóm.
cat_indices = list(range(len(NUMERIC), len(NUMERIC) + len(CATEGORICAL)))
smote = ImbPipeline([("prep", build_preprocessor("tree")),
                     ("sample", SMOTENC(categorical_features=cat_indices,
                                         sampling_strategy=0.75, random_state=SEED, k_neighbors=5)),
                     ("clf", classifier())])
smote.fit(X_train, y_train)
p_smote = smote.predict_proba(X_val)[:, 1]
m_smote = metrics_with_brier(y_val, p_smote)
log_result(AUTHOR, "E2", "lightgbm", "val", m_smote, "SMOTENC_ratio=0.75_train_only")
save_proba("lightgbm_smotenc", p_smote)

comparison = pd.DataFrame({"gốc": m_e1, "class_weight": m_weighted, "SMOTENC": m_smote}).T
comparison[["roc_auc", "pr_auc", "precision", "recall", "f1", "brier", "log_loss"]].round(4)

In [ ]:
comparison[["roc_auc", "pr_auc", "f1", "brier", "log_loss"]].to_csv(
    REPO_ROOT / "reports" / "nguoi3_e2_summary.csv", index_label="method")
print("Theo Brier nhỏ nhất:", comparison.brier.idxmin())
print("Theo F1 tại ngưỡng 0,5:", comparison.f1.idxmax())
print("Brier của bản gốc và cân bằng lớp giúp quyết định nguồn xác suất cho E7/E8.")

## E7 — Hiệu chỉnh xác suất trên phần cuối của train

Đặt mốc theo **ngày đến**, tránh cùng một ngày xuất hiện ở cả phần fit và phần hiệu chỉnh. Ngày đến chỉ dùng để chia tập, không được đưa vào đặc trưng. So sánh hai xác suất trước và sau hiệu chỉnh trên cùng val. Brier và biểu đồ calibration đo chất lượng xác suất; ROC-AUC đo thứ hạng.

In [ ]:
months = {m:i for i,m in enumerate(["January", "February", "March", "April", "May", "June",
     "July", "August", "September", "October", "November", "December"], 1)}
dates = pd.to_datetime(dict(year=X_train.arrival_date_year,
                            month=X_train.arrival_date_month.map(months),
                            day=X_train.arrival_date_day_of_month))
cutoff = dates.sort_values().iloc[int(len(dates) * 0.80)]
fit_mask = dates < cutoff
assert 0.15 < (~fit_mask).mean() < 0.25
base_cal = Pipeline([("prep", build_preprocessor("tree")), ("clf", classifier())])
base_cal.fit(X_train.loc[fit_mask], y_train.loc[fit_mask])
calibrated = fit_calibrator(base_cal, X_train.loc[~fit_mask], y_train.loc[~fit_mask])
p_uncal = base_cal.predict_proba(X_val)[:, 1]
p_cal = calibrated.predict_proba(X_val)[:, 1]
m_uncal, m_cal = metrics_with_brier(y_val, p_uncal), metrics_with_brier(y_val, p_cal)
log_result(AUTHOR, "E7", "lightgbm", "val", m_uncal, f"uncalibrated_fit_before_{cutoff.date()}")
log_result(AUTHOR, "E7", "lightgbm", "val", m_cal, f"sigmoid_calibrated_train_tail_from_{cutoff.date()}")
save_proba("lightgbm_calibrated", p_cal)
joblib.dump(calibrated, REPO_ROOT / "models" / "lightgbm_calibrated.joblib")
print(f"Fit: {fit_mask.sum():,} đơn trước {cutoff.date()}; hiệu chỉnh: {(~fit_mask).sum():,} đơn từ {cutoff.date()}")
pd.DataFrame({"trước hiệu chỉnh": m_uncal, "sau hiệu chỉnh": m_cal, "E1 toàn train": m_e1}).T[
    ["roc_auc", "pr_auc", "brier", "log_loss", "precision", "recall", "f1"]].round(4)

In [ ]:
from sklearn.calibration import CalibrationDisplay
fig, ax = plt.subplots(figsize=(7, 5))
for label, p in [("Trước", p_uncal), ("Sau sigmoid", p_cal)]:
    CalibrationDisplay.from_predictions(y_val, p, n_bins=10, strategy="quantile",
                                        name=label, ax=ax)
ax.set_title("LightGBM — hiệu chỉnh xác suất trên val")
fig.tight_layout()
fig.savefig(FIGURES / "nguoi3_e7_calibration.png", dpi=160)
plt.show()
pd.DataFrame({"model": ["before", "after"], "brier": [m_uncal["brier"], m_cal["brier"]],
              "log_loss": [m_uncal["log_loss"], m_cal["log_loss"]],
              "roc_auc": [m_uncal["roc_auc"], m_cal["roc_auc"]]}).to_csv(
    REPO_ROOT / "reports" / "nguoi3_e7_summary.csv", index=False)

## SHAP — Mô hình dựa vào đặc trưng nào?

Giải thích **mô hình E1 LightGBM**, mẫu val cố định 500 đơn. `agent` và `country` đã được OrdinalEncoder biến thành mã; SHAP với cột đó chỉ mô tả ảnh hưởng của mã mà mô hình dùng, không xem mã là đại lượng liên tục hay quan hệ nhân quả.

In [ ]:
rng = np.random.default_rng(SEED)
idx = np.sort(rng.choice(len(X_val), size=min(500, len(X_val)), replace=False))
Xs = X_val.iloc[idx]
encoded = model.named_steps["prep"].transform(Xs)
feature_names = model.named_steps["prep"].get_feature_names_out()
explainer = shap.TreeExplainer(model.named_steps["clf"])
shap_raw = explainer.shap_values(encoded)
# Với phân loại nhị phân, các phiên bản SHAP trả về ma trận hoặc danh sách hai ma trận.
if isinstance(shap_raw, list):
    values = shap_raw[1]
else:
    values = np.asarray(shap_raw)
    if values.ndim == 3:
        values = values[:, :, 1]
assert values.shape == encoded.shape
importance = pd.DataFrame({"feature": feature_names,
                           "mean_abs_shap": np.abs(values).mean(axis=0)}).sort_values(
                               "mean_abs_shap", ascending=False)
importance.to_csv(REPO_ROOT / "reports" / "nguoi3_shap_importance.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 6))
shap.summary_plot(values, encoded, feature_names=feature_names,
                  plot_type="bar", max_display=15, show=False)
plt.title("LightGBM: 15 đặc trưng quan trọng theo |SHAP|")
plt.tight_layout()
plt.savefig(FIGURES / "nguoi3_shap_importance.png", dpi=160, bbox_inches="tight")
plt.show()
importance.head(15)

In [ ]:
print("Lưu ý: đây là tác động lên dự đoán trong mô hình, không chứng minh nguyên nhân hủy.")
print("Chỉ số theo khách sạn trên val:")
display(evaluate_by_group(y_val, p_e1, X_val["hotel"]).round(4))
# Kiểm tra artifact E7 có thể nạp và dự đoán lại cùng thứ tự val.
reloaded = joblib.load(REPO_ROOT / "models" / "lightgbm_calibrated.joblib")
np.testing.assert_allclose(reloaded.predict_proba(X_val)[:, 1], p_cal, atol=1e-7)
print("Đã kiểm tra mô hình đã lưu và xác suất E7 khớp nhau.")

## Bàn giao cho nhóm

- `models/proba_val_lightgbm.npy`: E1, xác suất từ mô hình fit toàn bộ train.
- `models/proba_val_lightgbm_calibrated.npy`: E7, xác suất hiệu chỉnh, cùng thứ tự dòng `val.csv`. **Dùng cho E8** nếu Brier cải thiện; không tự ý đổi ngưỡng 0,5 thành ngưỡng chi phí nếu chưa tính chi phí.
- `models/lightgbm.joblib` và `models/lightgbm_calibrated.joblib`: mô hình tái sử dụng; cần có `src/calibration.py` khi nạp file hiệu chỉnh.
- `reports/results/nguoi3.csv`: các dòng E1, E2, E7; CSV tóm tắt và hình trong `reports/`.
- E9: `reports/nguoi3_shap_importance.csv` và hình SHAP; phần phân tích lỗi chi tiết là nhiệm vụ người 2.

**Giới hạn:** Dữ liệu chia theo ngày đến, không phải ngày đặt (không có cột ngày đặt). Một đặt phòng đến sau có thể đã được đặt trước một đơn đến sớm; vì vậy chưa thể khẳng định đánh giá hoàn toàn chống rò rỉ thời gian thực. Không dùng test cho tới khi cả nhóm chốt cuối.